In [1]:
import cv2
import glob
import torch
import numpy as np
from PIL import Image
from super_gradients.training import models


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES=['product']
CLASSES+=[str(i) for i in range(80 - len(CLASSES))]


model=models.get(
    "yolo_nas_s",
    num_classes=len(CLASSES),
    checkpoint_path=f"./weightsRe/KIKUYAIMG/average_model.pth"
)

[2024-03-19 16:38:58] INFO - crash_tips_setup.py - Crash tips is enabled. You can set your environment variable to CRASH_HANDLER=FALSE to disable it


The console stream is logged into /home/protiva/sg_logs/console.log


[2024-03-19 16:39:00] WARNING - __init__.py - Failed to import pytorch_quantization
[2024-03-19 16:39:00] WARNING - calibrator.py - Failed to import pytorch_quantization
[2024-03-19 16:39:00] WARNING - export.py - Failed to import pytorch_quantization
[2024-03-19 16:39:00] WARNING - selective_quantization_utils.py - Failed to import pytorch_quantization
[2024-03-19 16:39:00] INFO - checkpoint_utils.py - Successfully loaded model weights from ./weightsRe/KIKUYAIMG/average_model.pth EMA checkpoint.


In [2]:
models.convert_to_onnx(model=model,input_shape=(3,640,640),out_path="./weightsRe/KIKUYAIMG/average_model.onnx")


[2024-03-19 16:39:00] WARNING - conversion.py - input_shape is deprecated and will be removed in the next major release.Use the convert_to_onnx(..., prep_model_for_conversion_kwargs(input_size=(1, 3, 640, 640))) instead


'./weightsRe/KIKUYAIMG/average_model.onnx'

In [3]:
from yolo_nas_onnx.models import load_net
from yolo_nas_onnx.processing import Preprocessing, Postprocessing
from yolo_nas_onnx.draw import draw_box
from yolo_nas_onnx.utils import Labels


In [4]:
def detect(net,source,pre_process,post_process,labels):
    net_input = source.copy()#copying source array
    input_,prep_meta = pre_process(net_input)#running preprocessing
    outputs = net.forward(input_)#forward

    boxes,scores,classes = post_process(outputs,prep_meta)#post process
    selected = cv2.dnn.NMSBoxes(
        boxes,scores,post_process.score_thres,post_process.iou_thres
    )#nms to filter boxes

    for i in selected:
        box=boxes[i:].astype(np.int32).flatten()#get boxes
        score=float(scores[i])*100 #percent
        label,color = labels(classes[i],use_bgr=True)#label,color class id
        draw_box(source,box,label,score,color) #draw boxes
    return source


use_gpu = True
use_opencv_dnn_runtime = False
model_path="./weights/KIKUYAIMG/average_model.onnx"

net = load_net(model_path,use_gpu,use_opencv_dnn_runtime)
net.assert_input_shape([1,3,640,640])
net.warmup()
        

⚠️ GPU: GPU is not available, using CPU to process.


In [5]:
prep_steps = [
    {"DetLongMaxRescale": None},
    {"BotRightPad": {"pad_value": 114}}
]

iou_thres = 0.65
score_thres = 0.5
labels = ["0"]

_, _, input_height, input_width = net.input_shape  # get input height and width [b, c, h, w]

pre_process = Preprocessing(
    prep_steps, (input_height, input_width)
)

post_process = Postprocessing(
    prep_steps,
    iou_thres,
    score_thres,
)

labels = Labels(labels)

In [6]:
import cv2
import numpy as np
from PIL import Image

img = cv2.imread("./archive/resized_data/images/test/IMG_0457.jpg")
resized_image = cv2.resize(img,(640,640))
img = detect(net, resized_image, pre_process, post_process, labels)

Image.fromarray(img[:,:,::-1])

error: OpenCV(4.9.0) :-1: error: (-5:Bad argument) in function 'rectangle'
> Overload resolution failed:
>  - Can't parse 'pt1'. Expected sequence length 2, got 692
>  - Can't parse 'pt1'. Expected sequence length 2, got 692
>  - Can't parse 'rec'. Expected sequence length 4, got 692
>  - Can't parse 'rec'. Expected sequence length 4, got 692
